In [1]:
%load_ext line_profiler
%load_ext memory_profiler

In [2]:
import mmap
import time
import os
import struct
from ensembles import Config, Ensemble, EnsembleFormatError, EnsembleWriter
import gc

PATH = "200x.000"
batch_size = 4096

def scan_mmap(path):
    with open(path, "rb") as f:
        file_size = os.path.getsize(path)
        mm = mmap.mmap(f.fileno(), length=0, access=mmap.ACCESS_READ)

        current_offset = 0
        ens_indexes = []

        while current_offset < file_size:
            if current_offset + 4 > file_size:
                break  # not enough bytes left for a header
        
            header = mm[current_offset:current_offset + 4]
        
            if header[0] == 0x7f and header[1] == 0x7f:
                ens_size = header[2] + (header[3] << 8) + 2
                if 32 <= ens_size <= 4096 and current_offset + ens_size <= file_size:
                    ens_indexes.append(current_offset)
                    current_offset += ens_size
                    continue
        
            current_offset += 1

        return mm, ens_indexes

mm, ens_indexes = scan_mmap(PATH)

In [3]:
def decode_sample_ensembles(mm, ens_indexes):
    """Decode a sample of ensembles to profile performance"""
    # Get the length of an ensemble
    offset = ens_indexes[0]
    numbytes = struct.unpack("<h", mm[offset + 2:offset + 4])[0]

    # Decode the first ensemble to get config and structure data
    ens_dat = mm[offset:offset + numbytes]
    ens = Ensemble.from_bytes(ens_dat)

    if not Ensemble.config:
        raise EnsembleFormatError(
            "Configuration data missing from first ensemble")
    cfg = Ensemble.config

    # Needed for velocity shapes
    n_cells = Ensemble.config.n_cells

    # List for storing ensembles, add first ensemble
    batch = []
    batch.append(ens)

    # For all ensembles, create object and write batches to file
    for i in range(1, len(ens_indexes)):
        ens_dat = mm[ens_indexes[i]:ens_indexes[i] + numbytes]
        ens = Ensemble.from_bytes(ens_dat)
        batch.append(ens)

    return batch

In [28]:
%lprun -f Ensemble.from_bytes decode_sample_ensembles(mm, ens_indexes[:100000])

Timer unit: 1e-09 s

Total time: 3.53405 s
File: /home/zac/GitHub/COBIAlab-ADCP/ensembles.py
Function: from_bytes at line 105

Line #      Hits         Time  Per Hit   % Time  Line Contents
   105                                               @classmethod
   106                                               def from_bytes(cls, bytes):
   107    100000  657116910.0   6571.2     18.6          e = cls()
   108                                           
   109                                                   # Check that the header matches
   110    100000   21760552.0    217.6      0.6          if bytes[0] != 127 or bytes[1] != 127:
   111                                                       raise EnsembleFormatError("Header not at expected index")
   112                                           
   113                                                   # numbytes = bytes[2:4]
   114    100000   12641484.0    126.4      0.4          datatypes = bytes[5]
   115    100000   14919939.0    

In [5]:
%mprun -f Ensemble.from_bytes decode_sample_ensembles(mm, ens_indexes[:1000])

Filename: /home/zac/GitHub/COBIAlab-ADCP/ensembles.py

Line #    Mem usage    Increment  Occurrences   Line Contents
   105   7036.7 MiB   7036.5 MiB        1000       @classmethod
   106                                             def from_bytes(cls, bytes):
   107   7036.7 MiB      0.2 MiB        1000           e = cls()
   108                                         
   109                                                 # Check that the header matches
   110   7036.7 MiB      0.0 MiB        1000           if bytes[0] != 127 or bytes[1] != 127:
   111                                                     raise EnsembleFormatError("Header not at expected index")
   112                                         
   113                                                 # numbytes = bytes[2:4]
   114   7036.7 MiB      0.0 MiB        1000           datatypes = bytes[5]
   115   7036.7 MiB      0.0 MiB        1000           e.datatypes = datatypes
   116                                         

In [6]:
%prun decode_sample_ensembles(mm, ens_indexes[:1000])

         35679 function calls (35672 primitive calls) in 0.056 seconds

   Ordered by: internal time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
     1000    0.015    0.000    0.034    0.000 ensembles.py:105(from_bytes)
        1    0.013    0.013    0.034    0.034 history.py:845(writeout_cache)
    11000    0.008    0.000    0.008    0.000 {built-in method numpy.frombuffer}
      2/1    0.006    0.003    0.021    0.021 <string>:1(<module>)
     1000    0.003    0.000    0.007    0.000 <string>:2(__init__)
     7000    0.002    0.000    0.002    0.000 {built-in method numpy.zeros}
     4000    0.002    0.000    0.002    0.000 {method 'reshape' of 'numpy.ndarray' objects}
     1000    0.001    0.000    0.002    0.000 calendar.py:683(timegm)
       14    0.001    0.000    0.001    0.000 socket.py:626(send)
     1000    0.001    0.000    0.001    0.000 {method 'astype' of 'numpy.ndarray' objects}
        1    0.000    0.000    0.009    0.009 3511621106.py:1(de